# Testing detection.py

This notebook tests the detectors in `src/detection.py` to verify that
the rule-based flags produce reasonable counts before we add ML detectors.

## Setup

Load the features file once. Subsequent cells operate on `apc_features`
in memory.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
from src.detection import add_rule_flags

apc_features = pd.read_csv('../data/processed/apc_features.csv',
                            sep=';', encoding='latin-1')
print(f"Loaded apc_features: {apc_features.shape}")

Loaded apc_features: (1598277, 39)


## Verification: how the Halland bounding box was chosen

The bounds in `flag_gps_outside_halland` were not picked arbitrarily.
They are based on the actual GPS distribution in the January 2025 data.
We exclude the (0, 0) error rows first, then compute the 1st and 99th
percentile of latitude and longitude — meaning where 99% of valid
positions fall. The box used in the code is slightly wider than this
range, to leave margin for legitimate trips at the region's edges
without flagging them as anomalies.

In [2]:
# Filter out the (0,0) GPS errors so they don't skew the percentiles
real_gps = apc_features[(apc_features['latitude']  != 0) &
                        (apc_features['longitude'] != 0)]

print(f"Rows with valid (non-zero) GPS: {len(real_gps):,}")
print()

print("Latitude:")
print(f"  observed range:   {real_gps['latitude'].min():.4f} - {real_gps['latitude'].max():.4f}")
print(f"  1st - 99th pct:   {real_gps['latitude'].quantile(0.01):.4f} - {real_gps['latitude'].quantile(0.99):.4f}")
print(f"  bounds used:      56.1 - 57.5")
print()
print("Longitude:")
print(f"  observed range:   {real_gps['longitude'].min():.4f} - {real_gps['longitude'].max():.4f}")
print(f"  1st - 99th pct:   {real_gps['longitude'].quantile(0.01):.4f} - {real_gps['longitude'].quantile(0.99):.4f}")
print(f"  bounds used:      12.0 - 13.7")
print()

# Show what would be flagged
out_of_box = real_gps[~(real_gps['latitude'].between(56.1, 57.5) &
                       real_gps['longitude'].between(12.0, 13.7))]
print(f"Rows outside the box (and not (0,0)): {len(out_of_box):,}")
print()
print("Sample:")
print(out_of_box[['vehicleCode', 'arrival', 'latitude', 'longitude']].head().to_string(index=False))

Rows with valid (non-zero) GPS: 1,579,878

Latitude:
  observed range:   0.0833 - 57.4255
  1st - 99th pct:   56.4665 - 57.3132
  bounds used:      56.1 - 57.5

Longitude:
  observed range:   11.8627 - 16.1451
  1st - 99th pct:   12.1288 - 13.2413
  bounds used:      12.0 - 13.7

Rows outside the box (and not (0,0)): 10

Sample:
 vehicleCode             arrival  latitude  longitude
        4237 02/01/2025 19:02:09 55.750600  16.128400
        4237 02/01/2025 19:03:03 55.740330  16.129860
        4237 02/01/2025 19:05:23 55.708510  16.125948
        4237 02/01/2025 19:06:45 55.692930  16.127388
        4237 02/01/2025 19:07:36 55.680668  16.131268


## Run rule-based detectors

Apply all four rule-based detectors to the features data and count flags
both at row level (GPS rules) and trip level (load rules).

In [3]:
flagged = add_rule_flags(apc_features)
print(f"After rules: {flagged.shape}")
print()

print("Row-level flags:")
print(f"  flag_gps_zero:            {flagged['flag_gps_zero'].sum():,}")
print(f"  flag_gps_outside_halland: {flagged['flag_gps_outside_halland'].sum():,}")
print()

trips = flagged.dropna(subset=['trip']).drop_duplicates(['vehicleCode', 'trip', 'date'])
print(f"Total trips: {len(trips):,}")
print()
print("Trip-level flags:")
print(f"  flag_negative_load:     {trips['flag_negative_load'].sum():,}")
print(f"  flag_capacity_exceeded: {trips['flag_capacity_exceeded'].sum():,}")

After rules: (1598277, 43)

Row-level flags:
  flag_gps_zero:            18,399
  flag_gps_outside_halland: 18,409

Total trips: 64,172

Trip-level flags:
  flag_negative_load:     24,943
  flag_capacity_exceeded: 1


## Run ML detector: Isolation Forest

Run the generic `add_ml_flags` function with an Isolation Forest model.
The function works with any sklearn estimator that has `fit()` and
`score_samples()`, so switching to LOF or another algorithm is one line.
We flag the 5% most anomalous trips.

In [4]:
from sklearn.ensemble import IsolationForest
from src.detection import add_ml_flags

flagged = add_ml_flags(
    flagged,
    model=IsolationForest(contamination=0.05, random_state=42),
    name='if',
    threshold_pct=5,
)

print(f"After ML: {flagged.shape}")
print()

# Trip-level summary
trips = flagged.dropna(subset=['trip']).drop_duplicates(['vehicleCode', 'trip', 'date'])
print(f"Total trips: {len(trips):,}")
print()
print("ML-flagged trips:")
print(f"  flag_if: {trips['flag_if'].sum():,}  "
      f"({trips['flag_if'].sum() / len(trips) * 100:.1f}%)")

After ML: (1598277, 45)

Total trips: 64,172

ML-flagged trips:
  flag_if: 3,209  (5.0%)


## Compare rule-based vs ML detection

The examiner pointed out that anomalies detectable by simple thresholds
are not very interesting. The real value of an ML detector is finding
trips that look unusual in the *combination* of features but don't
trigger any single threshold rule.

We compare four groups:
- Trips flagged by ML only (the interesting set)
- Trips flagged by rules only (sanity check that ML doesn't miss obvious cases)
- Trips flagged by both
- Trips flagged by neither

In [5]:
# Combine all rule flags into one "any rule fired" column.
# Trip-level rules: negative_load, capacity_exceeded.
# We do not include GPS rules here since those are row-level (a trip with
# one bad GPS row is not necessarily an anomaly trip).
trips['flag_rules'] = trips['flag_negative_load'] | trips['flag_capacity_exceeded']

# Cross-tabulation
n_total      = len(trips)
n_rules_only = ((trips['flag_rules']) & (~trips['flag_if'])).sum()
n_ml_only    = ((~trips['flag_rules']) & (trips['flag_if'])).sum()
n_both       = ((trips['flag_rules']) & (trips['flag_if'])).sum()
n_neither    = ((~trips['flag_rules']) & (~trips['flag_if'])).sum()

print(f"Total trips: {n_total:,}")
print()
print(f"  Rules only:  {n_rules_only:>6,}  ({n_rules_only/n_total*100:5.1f}%)")
print(f"  ML only:     {n_ml_only:>6,}  ({n_ml_only/n_total*100:5.1f}%)")
print(f"  Both:        {n_both:>6,}  ({n_both/n_total*100:5.1f}%)")
print(f"  Neither:     {n_neither:>6,}  ({n_neither/n_total*100:5.1f}%)")
print()
print(f"Total flagged by rules: {trips['flag_rules'].sum():,}")
print(f"Total flagged by ML:    {trips['flag_if'].sum():,}")

Total trips: 64,172

  Rules only:  24,944  ( 38.9%)
  ML only:      1,328  (  2.1%)
  Both:         1,881  (  2.9%)
  Neither:     39,228  ( 61.1%)

Total flagged by rules: 24,944
Total flagged by ML:    3,209


## What does ML find that rules miss?

Inspect the ML-only trips: those flagged by the Isolation Forest but
not by any rule. We sort by ML score (most anomalous first) and look
at their feature values. If ML is doing something useful, these trips
should look subtly unusual — combinations of features that no single
threshold would catch.

In [6]:
ml_only = trips[(trips['flag_if']) & (~trips['flag_rules'])].copy()
ml_only = ml_only.sort_values('score_if')   # lowest score = most anomalous

print(f"ML-only trips: {len(ml_only):,}")
print()
print("Top 10 most anomalous ML-only trips:")
cols = ['vehicleCode', 'trip', 'date', 'line',
        'trip_total_boardings', 'trip_total_alightings',
        'trip_imbalance', 'trip_n_stops',
        'trip_min_load', 'trip_max_load', 'trip_imbalance_z_score',
        'score_if']
print(ml_only[cols].head(10).to_string(index=False))

ML-only trips: 1,328

Top 10 most anomalous ML-only trips:
 vehicleCode  trip       date   line  trip_total_boardings  trip_total_alightings  trip_imbalance  trip_n_stops  trip_min_load  trip_max_load  trip_imbalance_z_score  score_if
        3630  31.0 2025-01-13  652.0                  90.0                   46.0            44.0          28.0           20.0           63.0                5.281303 -0.768594
        3630  45.0 2025-01-31  615.0                 112.0                   33.0            79.0          57.0            5.0           79.0                7.370621 -0.766347
        3630  26.0 2025-01-29  615.0                  90.0                   55.0            35.0          59.0           11.0           71.0                3.152535 -0.765999
        7989  10.0 2025-01-30  350.0                 111.0                   64.0            47.0          42.0            0.0           86.0                9.120576 -0.765999
        3630   5.0 2025-01-09  614.0                  96.0   